# 02 — LigandMPNN (hairpin designs)

This notebook prepares LigandMPNN jobs for the RFD3 hairpin designs. It:
- Collects RFD3 output CIFs (uses CIF files directly, no PDB conversion)
- Extracts fixed residue regions from RFD3 config JSONs
- Creates a task CSV for LigandMPNN
- Builds a SLURM array script to run LigandMPNN with outputs to `/outputs/ligandmpnn/rfd3_hairpin`

In [2]:
import os
import json
import math
import csv
import gemmi
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

# =============================================================================
# 1) CONFIGURATION
# =============================================================================
PROJECT_ROOT = Path("/private/groups/yehlab/wsobolew/02_projects/computational/DR3M41_redesign_2026")
RFD3_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "rfd3_hairpin_020826"
LIGANDMPNN_DIR = Path("/private/groups/yehlab/wsobolew/01_software/LigandMPNN")

# Create output directory with timestamp
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
MPNN_OUT_DIR = PROJECT_ROOT / "outputs" / "ligandmpnn" / "rfd3_hairpin_020826" / TIMESTAMP
PDB_CACHE_DIR = MPNN_OUT_DIR / "pdbs"  # Store converted PDBs here
MPNN_OUT_DIR.mkdir(parents=True, exist_ok=True)
PDB_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Settings
NUM_SEQUENCES = 8
SAMPLING_TEMP = 0.1
ARRAY_BATCH_SIZE = 256
REDESIGN_CHAIN = "A"
PARSE_CHAINS = "A B" 
DEFAULT_FIXED_RESIDUES = [] # Empty list defaults to designing everything if config missing

In [3]:
# =============================================================================
# 2) HELPER FUNCTIONS
# =============================================================================

def convert_cif_to_pdb_gemmi(cif_path: Path, pdb_out_path: Path) -> bool:
    """
    Converts CIF (or .cif.gz) to PDB using Gemmi.
    Gemmi handles .gz transparently and is faster/more robust than BioPython.
    """
    try:
        # gemmi.cif.read extracts data from .cif or .cif.gz automatically
        doc = gemmi.cif.read(str(cif_path))
        
        # RFDiffusion CIFs usually have one block
        block = doc.sole_block()
        
        # Convert CIF block to Structure
        structure = gemmi.make_structure_from_block(block)
        
        # Write to PDB
        structure.write_pdb(str(pdb_out_path))
        return True
    except Exception as e:
        print(f"Gemmi conversion failed for {cif_path.name}: {e}")
        return False

def extract_fixed_residue_labels(config_path: Path) -> list:
    """
    Extract fixed residue labels using diffused_index_map values (output residue IDs).
    """
    try:
        with open(config_path, "r") as f:
            data = json.load(f)
        diffused_index_map = data.get("diffused_index_map", {})
        if not diffused_index_map:
            return []
        # Use values (output residue IDs), preserving order and de-duplicating
        seen = set()
        labels = []
        for v in diffused_index_map.values():
            if v not in seen:
                labels.append(v)
                seen.add(v)
        return labels
    except Exception:
        return []

In [4]:
# =============================================================================
# 3) PROCESSING LOOP
# =============================================================================
import multiprocessing

# Define worker function for multiprocessing
def process_single_design(item):
    stem, file_path = item
    
    # A. Convert to PDB using Gemmi
    pdb_out = PDB_CACHE_DIR / f"{stem}.pdb"
    
    # Gemmi handles .gz natively, so we pass file_path directly
    if not pdb_out.exists():
        success = convert_cif_to_pdb_gemmi(file_path, pdb_out)
        if not success:
            return None
    
    # B. Extract Fixed Residues for LigandMPNN (use original residue labels)
    config_path = RFD3_OUTPUT_DIR / f"{stem}.json"
    
    if config_path.exists():
        fixed_labels = extract_fixed_residue_labels(config_path)
    else:
        fixed_labels = DEFAULT_FIXED_RESIDUES
        
    # Convert to LigandMPNN fixed_residues string: "A1 A2 A3"
    if fixed_labels:
        fixed_residues_str = " ".join(fixed_labels)
    else:
        fixed_residues_str = ""

    # Return Task Data
    return {
        "name": stem,
        "pdb_path": str(pdb_out),
        "fixed_residues": fixed_residues_str
    }

# Get unique files
cif_files = sorted(list(RFD3_OUTPUT_DIR.glob("*.cif*")))

# Simple deduplication by stem
unique_files = {}
for f in cif_files:
    # Handle .cif and .cif.gz
    stem = f.name.replace(".cif.gz", "").replace(".cif", "")
    if stem not in unique_files:
        unique_files[stem] = f

print(f"Found {len(unique_files)} unique designs to process.")

# Multiprocessing Setup
NUM_CPUS = min(multiprocessing.cpu_count(), 32)  # Cap at 32 to avoid overloading shared nodes
print(f"Using {NUM_CPUS} CPUs for processing...")

tasks = []
items = list(unique_files.items())

# Run in parallel
with multiprocessing.Pool(processes=NUM_CPUS) as pool:
    # imap allows tqdm to show progress as items complete
    for result in tqdm(pool.imap(process_single_design, items), total=len(items), desc="Converting & Parsing"):
        if result is not None:
            tasks.append(result)

print(f"Successfully processed {len(tasks)} tasks.")

Found 20000 unique designs to process.
Using 32 CPUs for processing...


Converting & Parsing:   0%|          | 0/20000 [00:00<?, ?it/s]

Successfully processed 20000 tasks.


In [5]:
# =============================================================================
# 4) WRITE OUTPUTS
# =============================================================================

# Write CSV (Path map for SLURM array)
csv_path = MPNN_OUT_DIR / "mpnn_tasks.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    # Columns: PDB_PATH, NAME, FIXED_RESIDUES
    for t in tasks:
        writer.writerow([t["pdb_path"], t["name"], t["fixed_residues"]])

print(f"✅ Prepared {len(tasks)} tasks.")
print(f"📄 Task list written to: {csv_path}")

✅ Prepared 20000 tasks.
📄 Task list written to: /private/groups/yehlab/wsobolew/02_projects/computational/DR3M41_redesign_2026/outputs/ligandmpnn/rfd3_hairpin_020826/20260212_132203/mpnn_tasks.csv


In [6]:
# =============================================================================
# 5) BUILD SLURM ARRAY SCRIPT
# =============================================================================
slurm_path = MPNN_OUT_DIR / "run_ligandmpnn_array.sbatch"
log_dir = MPNN_OUT_DIR / "logs"
log_dir.mkdir(parents=True, exist_ok=True)
total_designs = len(tasks)
num_array_jobs = math.ceil(total_designs / ARRAY_BATCH_SIZE) if total_designs > 0 else 1

script_content = f"""#!/bin/bash
#SBATCH --job-name=DR3M41_mpnn
#SBATCH --error={log_dir}/error_%A_%a.err
#SBATCH --output={log_dir}/output_%A_%a.out
#SBATCH --time=04:00:00
#SBATCH --partition=gpu
#SBATCH --gres=gpu:A5500:1
#SBATCH --cpus-per-task=2
#SBATCH --mem=8G
#SBATCH --array=1-{num_array_jobs}%16

eval "$(micromamba shell hook --shell=bash)"
micromamba activate ligandmpnn_env

TASK_FILE="{csv_path}"
LIGANDMPNN_DIR="{LIGANDMPNN_DIR}"
OUT_DIR="{MPNN_OUT_DIR}"

# Checkpoints
CHECKPOINT_PATH="$LIGANDMPNN_DIR/model_params/ligandmpnn_v_32_010_25.pt"
SC_CHECKPOINT_PATH="$LIGANDMPNN_DIR/model_params/ligandmpnn_sc_v_32_002_16.pt"

START_LINE=$(( ($SLURM_ARRAY_TASK_ID - 1) * {ARRAY_BATCH_SIZE} + 1 ))
END_LINE=$(( $START_LINE + {ARRAY_BATCH_SIZE} - 1 ))

# Read Task File (PDB_PATH, NAME, FIXED_RESIDUES)
sed -n "${{START_LINE}},${{END_LINE}}p" "$TASK_FILE" | while IFS=$'\\t' read -r PDB_PATH NAME FIXED_RESIDUES; do
    if [ -z "$PDB_PATH" ]; then continue; fi
    
    echo "Running design for $NAME"

    python "$LIGANDMPNN_DIR/run.py" \\
        --pdb_path "$PDB_PATH" \\
        --out_folder "$OUT_DIR" \\
        --fixed_residues "$FIXED_RESIDUES" \\
        --chains_to_design "{REDESIGN_CHAIN}" \\
        --parse_these_chains_only "{PARSE_CHAINS}" \\
        --temperature {SAMPLING_TEMP} \\
        --number_of_batches {NUM_SEQUENCES} \\
        --batch_size 1 \\
        --model_type "ligand_mpnn" \\
        --checkpoint_ligand_mpnn "$CHECKPOINT_PATH" \\
        --checkpoint_path_sc "$SC_CHECKPOINT_PATH" \\
        --save_stats 1 \\
        --pack_side_chains 1 \\
        --pack_with_ligand_context 1

done
"""

with open(slurm_path, 'w') as f:
    f.write(script_content)

print(f"🚀 SLURM script written to: {slurm_path}")

🚀 SLURM script written to: /private/groups/yehlab/wsobolew/02_projects/computational/DR3M41_redesign_2026/outputs/ligandmpnn/rfd3_hairpin_020826/20260212_132203/run_ligandmpnn_array.sbatch


In [7]:
!sbatch {slurm_path}

Submitted batch job 28954423
